__Exploratory Data Analysis (EDA)__

- Inspect distributions, missing values, outliers.
- Plot time series of IV, skew, curvature.
- Compare SPY vs QQQ.
- Correlation checks.
- Document findings.

In [1]:
# import parquet_extractor  
# import importlib

# importlib.reload(parquet_extractor) 

In [2]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import polars as pl
import numpy as np
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from parquet_extractor import load_ticker_year_data, get_ticker_metadata

__Load the filtered parquet__

In [3]:
# metadata = get_ticker_metadata('SPY', Path("."))
# metadata

In [4]:
def load_merged_ticker_data(tickers, start_year, end_year, data_dir=Path(".")):
    """Load and merge data for multiple tickers across a range of years."""
    all_data = []

    if isinstance(tickers, str):
        tickers = [tickers]

    for ticker in tickers:
        for year in range(start_year, end_year + 1):
            try:
                df = load_ticker_year_data(ticker, year, data_dir)
                df = df.with_columns([
                    pl.lit(ticker).alias("ticker"),
                ])
                all_data.append(df)
            except FileNotFoundError:
                print(f"Data not found for {ticker} in {year}, skipping.")
            except Exception as e:
                print(f"Error loading {ticker} in {year}: {e}")

    if all_data:
        return pl.concat(all_data)
    else:
        return pl.DataFrame()

In [5]:
spy_df = load_merged_ticker_data('SPY', 2023, 2023, Path(".")).to_pandas()
spy_df.sample(7)

,date,secid,symbol,cp_flag,exdate,strike_price,best_bid,best_offer,volume,open_interest,...,price_diff_5d,price_diff_8d,price_diff_13d,price_diff_21d,price_diff_34d,price_diff_55d,price_diff_89d,price_diff_144d,price_diff_233d,ticker
1260114,2023-05-19,109820.0,SPY 231020C435000,C,2023-10-20,435000.0,11.49,11.54,149.0,1880.0,...,-0.61001,-0.61001,-0.61001,3.38999,8.37000,5.60999,5.76999,10.60001,12.54001,SPY
1356126,2023-05-05,109820.0,SPY 240315P445000,P,2024-03-15,445000.0,38.19,39.86,5.0,4964.0,...,0.00000,0.00000,7.50000,7.50000,1.79000,-3.29999,0.75000,4.58001,8.92999,SPY
1525321,2023-06-06,109820.0,SPY 240315C475000,C,2024-03-15,475000.0,7.21,7.48,6.0,899.0,...,0.92999,0.92999,0.10999,6.20999,7.85001,13.94000,17.78000,17.19000,15.57001,SPY
743119,2023-05-17,109820.0,SPY 241220C395000,C,2024-12-20,395000.0,61.48,63.49,5.0,4243.0,...,4.98001,4.98001,3.64001,3.10001,2.38000,10.10001,-0.69998,1.02002,11.53000,SPY
1333019,2023-08-03,109820.0,SPY 230915P443000,P,2023-09-15,443000.0,5.49,5.51,2430.0,20177.0,...,-1.29000,-1.29000,-1.29000,-7.64001,-8.95001,-3.64999,-6.60000,-6.36001,6.38001,SPY
703527,2023-04-12,109820.0,SPY 230519C393000,C,2023-05-19,393000.0,20.64,21.06,4.0,6706.0,...,-1.67001,-1.67001,-1.67001,-1.56000,-1.14001,0.44998,-1.34002,11.56000,11.94000,SPY
1197762,2023-02-10,109820.0,SPY 230317C430000,C,2023-03-17,430000.0,1.78,1.80,2266.0,36070.0,...,0.95001,0.95001,0.95001,-2.60998,-7.14999,-4.31000,1.56000,7.69000,11.08002,SPY


In [8]:
qqq_pl = load_merged_ticker_data('QQQ', 2005, 2023, Path("."))
qqq_df = qqq_pl.to_pandas()
qqq_df.sample(7)

,date,secid,symbol,cp_flag,exdate,strike_price,best_bid,best_offer,volume,open_interest,...,price_diff_5d,price_diff_8d,price_diff_13d,price_diff_21d,price_diff_34d,price_diff_55d,price_diff_89d,price_diff_144d,price_diff_233d,ticker
2799105,2018-06-25,107899.0,QQQ 181221P168000,P,2018-12-21,168000.0,7.46,7.56,9.0,340.0,...,0.00000,0.00000,-3.95001,-5.88000,-4.63000,-6.23001,-1.37001,2.39000,11.24000,QQQ
2603898,2018-08-01,107899.0,QQQ 180921P145000,P,2018-09-21,145000.0,0.26,0.29,31.0,113830.0,...,0.67000,2.01000,2.01000,-0.50000,-2.93000,-2.44000,-3.15000,6.32000,-0.48001,QQQ
3702487,2020-10-15,107899.0,QQQ 201218C206000,C,2020-12-18,206000.0,84.55,84.75,4.0,267.0,...,-4.41998,-4.42999,4.39002,18.54001,17.62000,19.65000,20.72001,30.11002,70.76001,QQQ
4965120,2021-05-24,107899.0,QQQ 210528C350000,C,2021-05-28,350000.0,0.02,0.03,778.0,6006.0,...,5.50000,5.50000,3.68002,3.68002,10.29001,13.17001,-1.69000,-7.70999,-9.09998,QQQ
1207251,2012-11-20,107899.0,QQQ 130316P58000,P,2013-03-16,58000.0,1.01,1.04,1600.0,18529.0,...,0.01940,0.01940,0.01940,1.50000,1.56000,0.38000,-1.92000,-2.22000,-4.77000,QQQ
3099208,2019-04-01,107899.0,QQQ 190621P162000,P,2019-06-21,162000.0,1.04,1.06,25.0,9431.0,...,0.00000,2.37999,2.37999,3.72999,4.14000,3.47999,3.58999,10.61000,10.41999,QQQ
4471279,2021-03-18,107899.0,QQQ 210618C265000,C,2021-06-18,265000.0,51.70,51.99,11.0,3927.0,...,0.00000,0.00000,-9.85998,-6.78998,-3.41998,12.10001,-11.54999,-20.42999,-10.38000,QQQ


In [9]:
# Save to Parquet
qqq_pl.write_parquet("./parquet/qqq_historical_2005_2023_v1.parquet")

In [17]:
qqq_df.sample(21)

,date,secid,symbol,cp_flag,exdate,strike_price,best_bid,best_offer,volume,open_interest,...,price_diff_5d,price_diff_8d,price_diff_13d,price_diff_21d,price_diff_34d,price_diff_55d,price_diff_89d,price_diff_144d,price_diff_233d,ticker
6653588,2023-04-21,107899.0,QQQ 230721P314000,P,2023-07-21,314000.0,10.88,10.92,33.0,2436.0,...,0.32999,-2.10000,-2.25000,-2.23001,-1.96002,3.56998,0.68998,7.84998,22.57999,QQQ
1401118,2013-06-10,107899.0,QQQ 130817C68000,C,2013-08-17,68000.0,5.68,5.76,1.0,602.0,...,0.00000,0.00000,0.03000,1.02000,1.37000,0.17000,-0.19900,0.37000,3.63000,QQQ
6234585,2022-08-25,107899.0,QQQ 240119C390000,C,2024-01-19,390000.0,16.68,17.40,1.0,733.0,...,-12.48001,-9.81002,-9.81002,-3.50000,-0.13000,10.76999,28.70999,36.78000,13.35999,QQQ
1957675,2015-10-13,107899.0,QQQ 160318P113000,P,2016-03-18,113000.0,9.39,9.63,85.0,1962.0,...,0.05000,0.05000,0.47000,2.09000,3.18000,0.42000,2.71000,-0.98000,-5.03000,QQQ
4319050,2020-12-23,107899.0,QQQ 210115P307000,P,2021-01-15,307000.0,6.40,6.42,771.0,5174.0,...,0.00000,0.00000,-1.56000,-0.72000,-2.79999,0.91000,5.68002,4.35000,18.09002,QQQ
1099327,2011-05-13,107899.0,QQQ 111217P57000,P,2011-12-17,57000.0,3.24,3.30,10.0,3826.0,...,-0.70000,-0.70000,-0.34500,-0.78000,0.13000,-0.81000,0.53500,1.65000,1.33000,QQQ
5769205,2022-06-07,107899.0,QQQ 231215P310000,P,2023-12-15,310000.0,36.27,36.93,17.0,4401.0,...,2.65000,2.65000,3.66999,-4.51000,3.87000,22.63001,20.29001,18.03000,-15.52999,QQQ
1108733,2011-06-24,107899.0,QQQ 110820C58000,C,2011-08-20,58000.0,0.31,0.32,766.0,29261.0,...,-0.96000,-0.96000,-0.45100,-0.45100,0.31000,0.30000,-0.26000,-2.83000,-3.83000,QQQ
2358138,2017-10-27,107899.0,QQQ 171215P130000,P,2017-12-15,130000.0,0.16,0.17,432.0,50877.0,...,4.28000,4.28000,3.81002,3.25000,3.50000,2.93001,3.20002,5.66001,5.68001,QQQ
336101,2007-10-08,107899.0,QQQ.XQ,P,2007-12-22,43000.0,0.14,0.17,190.0,59871.0,...,0.33200,0.33200,0.33200,1.50000,1.14000,1.57000,3.12000,4.22000,4.95000,QQQ


In [15]:
metadata = get_ticker_metadata('QQQ', Path("."), from_date='2005-01-01')
metadata

TypeError: get_ticker_metadata() got an unexpected keyword argument 'from_date'